<a href="https://colab.research.google.com/github/Shanmuganathan75/QM640-WALSH-CAPSTONE/blob/main/04_yahoo_finance_pull.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QM 640 Capstone — Step 4: Return-Series Collection (Yahoo Finance)

For every screened, confirmed event, pulls daily stock returns for the
event firm and the S&P 500 market index (both cap-weighted and equal-
weighted, for the Synopsis's robustness check), covering the estimation
window through the long event window.

**Run this only after Step 3 (manual screening) and Step 3b (ticker
mapping) are both complete and pushed to the repo.**

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every
other cell in this notebook reads from and writes to.

In [ ]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "Shan_muganathan@yahoo.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 96, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 96 (delta 35), reused 77 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (96/96), 886.15 KiB | 2.71 MiB/s, done.
Resolving deltas: 100% (35/35), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [ ]:
!pip install -q pandas numpy yfinance

## Cell 3 — Configuration

In [ ]:
import os

SCREENED_FILE = os.path.join(BASE_DIR, "data/raw/screening_worksheet.csv")
OUTPUT_DIR = os.path.join(BASE_DIR, "data/processed/returns")
os.makedirs(OUTPUT_DIR, exist_ok=True)

MARKET_TICKER = "^GSPC"        # S&P 500 (cap-weighted, primary benchmark)
MARKET_TICKER_EW = "^SP500EW"  # S&P 500 Equal Weight (robustness check)

# Calendar-day buffer (Yahoo Finance indexes by calendar date, not trading day)
CALENDAR_DAYS_BEFORE = 230   # covers 150 trading days back + buffer
CALENDAR_DAYS_AFTER = 45     # covers the 30-day long event window + buffer

## Cell 4 — Load confirmed events (post-screening)

In [ ]:
import pandas as pd


def load_confirmed_events():
    df = pd.read_csv(SCREENED_FILE)
    df = df[df["is_genuine_ai_event"].astype(str).str.upper() == "Y"]
    df = df[df["confounding_event_flag"].astype(str).str.upper() != "Y"]
    df = df[df["trading_halt_flag"].astype(str).str.upper() != "Y"]
    df = df[df["sufficient_history_flag"].astype(str).str.upper() == "Y"]
    df = df.dropna(subset=["ticker"])  # ticker comes pre-merged from Step 3b
    df["file_date"] = pd.to_datetime(df["file_date"])
    df["event_id"] = df["accession_no"]
    print(f"Confirmed events after screening: {len(df)}")
    return df


events = load_confirmed_events()
events.head()

Confirmed events after screening: 51


,accession_no,query,cik,company_name,form_type,file_date,adsh,file_name,is_genuine_ai_event,announcement_type,confounding_event_flag,trading_halt_flag,sufficient_history_flag,exclude_reason,filing_url,screener_notes,ticker,item_codes,item_in_scope,event_id
611,0001907982-24-000015:d-wavezapataaixpressrele.htm,"""generative AI""",1907982,"D-Wave Quantum Inc. (QBTS, QBTS-WT) (CIK 000...",8-K,2024-02-08,0001907982-24-000015,NaN,Y,partnership,N,N,Y,NaN,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,QBTS,8.01,Y,0001907982-24-000015:d-wavezapataaixpressrele.htm
646,0001140361-24-008680:ef20022054_ex99-1.htm,"""generative AI""",1368514,"ADMA BIOLOGICS, INC. (ADMA) (CIK 0001368514)",8-K,2024-02-21,0001140361-24-008680,NaN,Y,R&D,N,N,Y,NaN,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,ADMA,"8.01,9.01",Y,0001140361-24-008680:ef20022054_ex99-1.htm
718,0000796343-24-000057:adbeex991q124.htm,"""generative AI""",796343,ADOBE INC. (ADBE) (CIK 0000796343),8-K,2024-03-14,0000796343-24-000057,NaN,Y,R&D,N,N,Y,NaN,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,ADBE,"2.02,8.01,9.01",Y,0000796343-24-000057:adbeex991q124.htm
805,0001193125-24-125080:d810346dex991.htm,"""generative AI""",1023313,"FORRESTER RESEARCH, INC. (FORR) (CIK 0001023...",8-K,2024-04-30,0001193125-24-125080,NaN,Y,R&D,N,N,Y,NaN,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,FORR,"2.02,8.01,9.01",Y,0001193125-24-125080:d810346dex991.htm
991,0001493152-24-028109:ex99-1.htm,"""generative AI""",64463,"Soluna Holdings, Inc (SLNH, SLNHP) (CIK 0000...",8-K,2024-07-17,0001493152-24-028109,NaN,Y,R&D,N,N,Y,NaN,https://www.sec.gov/cgi-bin/browse-edgar?actio...,NaN,SLNH,"1.01,2.03,3.02,7.01,9.01",Y,0001493152-24-028109:ex99-1.htm


## Cell 5 — Pull returns for each confirmed event

In [ ]:
import yfinance as yf
import time


def flatten_columns(data):
    """Recent yfinance returns multi-level columns even for a single ticker
    (e.g. ('Close', 'AAPL')), which breaks .rename() and other single-level
    operations downstream. Flatten to plain column names."""
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.droplevel(1)
    return data


def pull_returns_for_event(ticker, event_date, market_series, market_ew_series):
    start = event_date - pd.Timedelta(days=CALENDAR_DAYS_BEFORE)
    end = event_date + pd.Timedelta(days=CALENDAR_DAYS_AFTER)

    try:
        hist = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        hist = flatten_columns(hist)
    except Exception as e:
        print(f"  {ticker}: download failed ({e})")
        return None

    if hist.empty or len(hist) < 150:
        print(f"  {ticker}: insufficient history ({len(hist)} rows) - excluded per criteria (d)")
        return None

    hist["daily_return_firm"] = hist["Close"].pct_change()
    hist = hist.join(market_series.rename("daily_return_market"), how="inner")
    hist = hist.join(market_ew_series.rename("daily_return_market_ew"), how="left")

    n_before = (hist.index < event_date).sum()
    hist["trading_day_offset"] = range(-n_before, len(hist) - n_before)
    return hist[["daily_return_firm", "daily_return_market", "daily_return_market_ew",
                 "trading_day_offset"]].dropna(subset=["daily_return_firm"])


overall_start = events["file_date"].min() - pd.Timedelta(days=CALENDAR_DAYS_BEFORE)
overall_end = events["file_date"].max() + pd.Timedelta(days=CALENDAR_DAYS_AFTER)

print("Pulling market index series (S&P 500 cap-weighted + equal-weight) ...")
mkt_data = yf.download(MARKET_TICKER, start=overall_start, end=overall_end,
                        progress=False, auto_adjust=True)
mkt_data = flatten_columns(mkt_data)
mkt = mkt_data["Close"].pct_change()

mkt_ew_data = yf.download(MARKET_TICKER_EW, start=overall_start, end=overall_end,
                           progress=False, auto_adjust=True)
mkt_ew_data = flatten_columns(mkt_ew_data)
mkt_ew = mkt_ew_data["Close"].pct_change()

collected, skipped = 0, 0
for _, row in events.iterrows():
    df = pull_returns_for_event(row["ticker"], row["file_date"], mkt, mkt_ew)
    if df is None:
        skipped += 1
        continue

    out_path = os.path.join(OUTPUT_DIR, f"{row['ticker']}_{row['event_id']}.csv")
    df.to_csv(out_path)
    collected += 1
    time.sleep(0.2)  # be polite to Yahoo Finance

print(f"\nDone. Collected: {collected} | Skipped (excluded/failed): {skipped}")

Pulling market index series (S&P 500 cap-weighted + equal-weight) ...


/tmp/ipykernel_832/4294714014.py:46: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  mkt = mkt_data["Close"].pct_change()
/tmp/ipykernel_832/4294714014.py:29: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  hist["daily_return_firm"] = hist["Close"].pct_change()
/tmp/ipykernel_832/4294714014.py:29: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  hist["daily_return_firm"] = hist["Close"].pct_change()
/tmp/ip


Done. Collected: 51 | Skipped (excluded/failed): 0


## Commit and push results back to GitHub

In [ ]:
!git -C {BASE_DIR} add "data/processed/returns/"
!git -C {BASE_DIR} commit -m "Step 4: Yahoo Finance return series for confirmed events"
!git -C {BASE_DIR} push

[main 25e3e89] Step 4: Yahoo Finance return series for confirmed events
 51 files changed, 9558 insertions(+)
 create mode 100644 data/processed/returns/ABSI_0001672688-26-000003:finalabscijpm2026present.htm.csv
 create mode 100644 data/processed/returns/ADBE_0000796343-24-000057:adbeex991q124.htm.csv
 create mode 100644 data/processed/returns/ADMA_0001140361-24-008680:ef20022054_ex99-1.htm.csv
 create mode 100644 data/processed/returns/AITX_0001493152-24-036279:ex99-1.htm.csv
 create mode 100644 data/processed/returns/ALBT_0001213900-26-003178:ea027252201ex99-1_avalon.htm.csv
 create mode 100644 data/processed/returns/AMZN_0001104659-26-021050:tm267374d1_ex99-1.htm.csv
 create mode 100644 data/processed/returns/BGDE_0001171843-24-004941:exh_991.htm.csv
 create mode 100644 data/processed/returns/BNAI_0001641172-25-014420:ex99-1.htm.csv
 create mode 100644 data/processed/returns/CRNC_0001193125-24-284230:d915960dex991.htm.csv
 create mode 100644 data/processed/returns/FORR_0001193125-24